# Pure Press: train our own pair model on Colab

Trains the multilingual pair model (does this headline or sentence support, say nothing about, or contradict
this claim?) on the public sets (MNLI, FEVER, FNC-1, XNLI Hindi, IndicXNLI Tamil and Hindi), our synthetic
pairs, the claim-matching suites, and our own data: Google News headlines judged against forwarded claims,
labelled by Gemini through the verifier's stance step.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**, and upload
`pure_press_ml.zip` when asked. With mDeBERTa (the default below) it takes **about 2 to 2½ hours**: keep this
tab open and look at it now and then, since Colab may stop a session nobody is watching. The last cell
downloads `pair_model_export.zip`; put it in the project's `ml/` folder and tell Claude.

Nothing here touches the project or the server; the result is measured against the shipped model before it is used.

In [ ]:
# Settings. mDeBERTa-v3 (278M) reads meaning better than the MiniLM (118M) the server ships; its word table
# stays frozen (the multilingual one it was pre-trained with), which saves time and keeps Tamil and Hindi.
MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
EPOCHS = 2
LR = 2e-5
FREEZE_EMBEDDINGS = True

In [ ]:
# 1. A GPU is required
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then run again.'
print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())

In [ ]:
# 2. Upload pure_press_ml.zip (the training scripts, our labelled pairs and the shipped model for comparison)
import io, zipfile
from google.colab import files
uploaded = files.upload()
name = next(iter(uploaded))
zipfile.ZipFile(io.BytesIO(uploaded[name])).extractall('/content/pp')
%cd /content/pp
!ls

In [ ]:
# 3. Libraries (torch is already on Colab)
!pip install -q "transformers>=5" datasets accelerate scikit-learn sentencepiece protobuf tiktoken onnx onnxruntime onnxscript huggingface_hub pandas
import transformers; print('transformers', transformers.__version__)

In [ ]:
# 4. Data: the public sets, our synthetic pairs, then train / dev / evaluation files (about 5 minutes)
!python -X utf8 ml/src/fetch_data.py ml/data/raw
!python -X utf8 ml/src/synth.py --out ml/data
!cp stance_pairs.jsonl ml/data/
!python -X utf8 ml/src/build_data.py ml/data backend/eval/suites

In [ ]:
# 5. Fine-tune (the longest step: about 1½ to 2 hours for mDeBERTa on a T4; progress is printed every 200 steps,
#    and the weights are saved after each epoch)
freeze = '--freeze-embeddings' if FREEZE_EMBEDDINGS else ''
!python -X utf8 ml/src/train.py --model {MODEL} --data ml/data --out ml/artifacts/pair-model-ft --epochs {EPOCHS} --lr {LR} {freeze}

In [ ]:
# 6. Export to ONNX with a pruned vocabulary and int8 weights, as the server loads it
!python -X utf8 ml/src/export_onnx.py --model ml/artifacts/pair-model-ft --out ml/artifacts/pair-model-onnx-int8 --data ml/data
!ls -la ml/artifacts/pair-model-onnx-int8/onnx

In [ ]:
# 7. Measure: the new int8 model against the one the server ships, on the same held-out sets
!mkdir -p ml/artifacts/eval
!python -X utf8 ml/src/evaluate.py --model shipped_model --onnx shipped_model/onnx/model_quantized.onnx --data ml/data --out ml/artifacts/eval/shipped_int8.json
!python -X utf8 ml/src/evaluate.py --model ml/artifacts/pair-model-onnx-int8 --onnx ml/artifacts/pair-model-onnx-int8/onnx/model_quantized.onnx --data ml/data --out ml/artifacts/eval/new_int8.json
!python -X utf8 ml/src/tables.py "shipped=ml/artifacts/eval/shipped_int8.json" "new=ml/artifacts/eval/new_int8.json" | tee ml/artifacts/eval/comparison.md

In [ ]:
# 8. Download the export and the measurements
!cp ml/artifacts/pair-model-ft/train_log.json ml/artifacts/eval/ 2>/dev/null; true
!cd ml/artifacts && rm -f /content/pair_model_export.zip && zip -qr /content/pair_model_export.zip pair-model-onnx-int8 eval -x 'pair-model-onnx-int8/onnx/model.onnx'
!ls -la /content/pair_model_export.zip
files.download('/content/pair_model_export.zip')